In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/steam_clean.csv")
print(df.shape)

(20024, 20)


In [3]:

def success_tier(row):
    score = row['review_score']
    reviews = row['positive_ratings'] + row['negative_ratings']
    if score >= 0.85 and reviews >= 2000:
        return 'High'
    elif score >= 0.70 and reviews >= 200:
        return 'Medium'
    else:
        return 'Low'

df['success_tier'] = df.apply(success_tier, axis=1)

print(df['success_tier'].value_counts())




success_tier
Low       15341
Medium     3841
High        842
Name: count, dtype: int64


In [4]:
genres_encoded = df['genres'].str.get_dummies(sep=';')
genres_encoded.columns = ['genre_' + c for c in genres_encoded.columns]

print(genres_encoded.shape)
print(genres_encoded.head())

(20024, 27)
   genre_Accounting  genre_Action  genre_Adventure  \
0                 0             1                0   
1                 0             1                0   
2                 0             1                0   
3                 0             1                0   
4                 0             1                0   

   genre_Animation & Modeling  genre_Audio Production  genre_Casual  \
0                           0                       0             0   
1                           0                       0             0   
2                           0                       0             0   
3                           0                       0             0   
4                           0                       0             0   

   genre_Design & Illustration  genre_Early Access  genre_Education  \
0                            0                   0                0   
1                            0                   0                0   
2                      

In [5]:
# Encode top 20 tags as binary columns
tags_encoded = df['steamspy_tags'].str.get_dummies(sep=';')
tags_encoded.columns = ['tag_' + c for c in tags_encoded.columns]

# Keep only top 20 most common tags
top_20_tags = tags_encoded.sum().sort_values(ascending=False).head(20).index
tags_encoded = tags_encoded[top_20_tags]

print(tags_encoded.shape)

(20024, 20)


In [6]:
# Developer reputation — average score of all their previous games
dev_stats = df.groupby('developer')['review_score'].agg(['mean', 'count'])
dev_stats.columns = ['dev_avg_score', 'dev_game_count']
df = df.merge(dev_stats, on='developer', how='left')

# Log transform price
df['log_price'] = np.log1p(df['price'])

# Combine all features into one dataframe
final_df = pd.concat([
    df[['log_price', 'release_year', 'english', 'dev_avg_score', 'dev_game_count', 'success_tier']],
    genres_encoded,
    tags_encoded
], axis=1)

print(final_df.shape)
print(final_df.head())

(20024, 53)
   log_price  release_year  english  dev_avg_score  dev_game_count  \
0   2.102914          2000        1       0.890779              26   
1   1.607436          1999        1       0.890779              26   
2   1.607436          2003        1       0.890779              26   
3   1.607436          2001        1       0.890779              26   
4   1.607436          1999        1       0.819892               7   

  success_tier  genre_Accounting  genre_Action  genre_Adventure  \
0         High                 0             1                0   
1       Medium                 0             1                0   
2         High                 0             1                0   
3       Medium                 0             1                0   
4         High                 0             1                0   

   genre_Animation & Modeling  ...  tag_Racing  tag_Sports  tag_VR  \
0                           0  ...           0           0       0   
1                       

In [7]:
final_df.to_csv("../data/processed/steam_features.csv", index=False)
print("Saved!")

Saved!


In [8]:
# Add more features
df['has_achievements'] = (df['achievements'] > 0).astype(int)
df['platform_count'] = (
    df['platforms'].str.contains('windows').astype(int) +
    df['platforms'].str.contains('mac').astype(int) +
    df['platforms'].str.contains('linux').astype(int)
)
df['log_playtime'] = np.log1p(df['average_playtime'])

print(df[['has_achievements', 'platform_count', 'log_playtime']].head())

   has_achievements  platform_count  log_playtime
0                 0               3      9.776393
1                 0               3      5.627621
2                 0               3      5.236442
3                 0               3      5.556828
4                 0               3      6.437752


In [9]:
# Add new features to final_df
final_df = pd.concat([
    df[['log_price', 'release_year', 'english', 'dev_avg_score', 'dev_game_count', 
        'has_achievements', 'platform_count', 'log_playtime', 'success_tier']],
    genres_encoded,
    tags_encoded
], axis=1)

print(final_df.shape)

(20024, 56)


In [10]:
final_df.to_csv("../data/processed/steam_features.csv", index=False)
print("Saved!")


Saved!


['tag_Indie', 'tag_Action', 'tag_Adventure', 'tag_Casual', 'tag_Strategy', 'tag_Simulation', 'tag_RPG', 'tag_Early Access', 'tag_Free to Play', 'tag_Puzzle', 'tag_Racing', 'tag_Sports', 'tag_VR', 'tag_Platformer', 'tag_Anime', 'tag_Visual Novel', 'tag_Nudity', 'tag_Sexual Content', 'tag_Horror', 'tag_Point & Click']
